In [1]:
import sys
import os
from pathlib import Path
import pandas as pd
import numpy as np

# Connect to the ml directory and load the data
ml_root = Path().resolve().parents[0]
sys.path.append(str(ml_root))
from src.data_loader import load_train_data

train = load_train_data("FD001")

# Recalculate RUL from Day 2
max_cycle = train.groupby("engine_id")["cycle"].transform("max")
train["RUL"] = max_cycle - train["cycle"]

print("Data loaded and RUL calculated. Shape:", train.shape)

Data loaded and RUL calculated. Shape: (20631, 27)


In [2]:
# ==========================================================
# Phase 1 : Drop Useless Sensors (From Day 2 Analysis)
# ==========================================================
drop_sensors = [
    "sensor_1", "sensor_5", "sensor_6",
    "sensor_10", "sensor_16", "sensor_18", "sensor_19"
]
train_clean = train.drop(columns=drop_sensors)

print("=" * 60)
print(f"Dropped {len(drop_sensors)} zero-variance sensors.")

# ==========================================================
# Phase 2 : Piecewise RUL (Capping)
# ==========================================================
RUL_CAP = 125
train_clean["RUL"] = train_clean["RUL"].clip(upper=RUL_CAP)

print("-" * 60)
print("RUL Statistics After Capping (Max should be 125):")
print(train_clean["RUL"].describe()[['min', 'mean', 'max']])

# ==========================================================
# Phase 3 : Rolling Features (Time-Series Trends)
# ==========================================================
sensor_cols = [col for col in train_clean.columns if col.startswith("sensor_")]

def add_rolling_features(df, window=5):
    # Sort strictly by engine and time to prevent data leakage
    df = df.sort_values(["engine_id", "cycle"]).copy()

    for col in sensor_cols:
        # Calculate Rolling Mean (Trend)
        df[f"{col}_rollmean"] = (
            df.groupby("engine_id")[col]
            .transform(lambda x: x.rolling(window=window, min_periods=1).mean())
        )
        # Calculate Rolling Std (Instability)
        df[f"{col}_rollstd"] = (
            df.groupby("engine_id")[col]
            .transform(lambda x: x.rolling(window=window, min_periods=1).std())
        )
    return df

train_features = add_rolling_features(train_clean)
# Fill the first row's NaN standard deviation with 0
train_features = train_features.fillna(0) 

# ==========================================================
# Phase 4 : Sanity Checks & Saving
# ==========================================================
print("=" * 60)
print(f"Feature Engineering Complete. Final Shape: {train_features.shape}")
print(f"Missing Values: {train_features.isnull().sum().sum()}")

# Save to the processed folder
processed_dir = ml_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)
save_path = processed_dir / "train_FD001_features.csv"

train_features.to_csv(save_path, index=False)

print("-" * 60)
print(f"Processed dataset saved successfully to:\n{save_path}")

Dropped 7 zero-variance sensors.
------------------------------------------------------------
RUL Statistics After Capping (Max should be 125):
min       0.000000
mean     86.829286
max     125.000000
Name: RUL, dtype: float64
Feature Engineering Complete. Final Shape: (20631, 48)
Missing Values: 0
------------------------------------------------------------
Processed dataset saved successfully to:
/Users/princegupta/PredictX-AI/ml/data/processed/train_FD001_features.csv


In [3]:
# ----------------------------------------------------------
# Step 5 : Save Processed Dataset
# ----------------------------------------------------------

processed_dir = "../data/processed"

os.makedirs(processed_dir, exist_ok=True)

save_path = os.path.join(
    processed_dir,
    "train_FD001_features.csv"
)

train_features.to_csv(save_path, index=False)

print("\nProcessed dataset saved successfully.")
print(save_path)


# ----------------------------------------------------------
# Step 6 : Feature Summary
# ----------------------------------------------------------

print("\nRaw Sensors Used")

remaining_sensors = [
    c for c in sensor_cols
]

print(remaining_sensors)

print("\nRolling Mean Features")

print(len([c for c in train_features.columns if "rollmean" in c]))

print("\nRolling Std Features")

print(len([c for c in train_features.columns if "rollstd" in c]))

print("\nFinal Number of Features")

print(len(train_features.columns))


Processed dataset saved successfully.
../data/processed/train_FD001_features.csv

Raw Sensors Used
['sensor_2', 'sensor_3', 'sensor_4', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_17', 'sensor_20', 'sensor_21']

Rolling Mean Features
14

Rolling Std Features
14

Final Number of Features
48
